# Generative AI: From Prediction to Creation

In [1]:
"""
Ollama Meeting Notes Summariser
Uses a local LLM (via Ollama) to generate a structured summary of meeting notes.
"""

import requests
import sys

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "deepseek-r1:latest"   # Change to any model you have pulled

# Example meeting notes
MEETING_NOTES = """
Client wants faster onboarding. Current process takes 3 days.
Team suggested API automation for document verification.
Compliance approval is still mandatory. Pilot should start next month.
"""


def generate_summary(notes: str, model: str = MODEL_NAME) -> str:
    """
    Send meeting notes to Ollama and return a structured summary.

    Args:
        notes: Raw meeting notes text
        model: Ollama model name (must be already pulled)

    Returns:
        Generated summary string, or an error message if something fails.
    """
    prompt = f"""
You are a business analyst. Summarize the following meeting notes in exactly three sections:

1. Business Need
2. Proposed Approach
3. Next Step

Use simple business language and keep each section concise.

Meeting Notes:
{notes}
"""

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0.2}   # lower temperature = more deterministic
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=60)
        response.raise_for_status()          # Raise an error for bad HTTP status

        data = response.json()
        # The response structure for non‑streaming chat:
        summary = data["message"]["content"]
        return summary.strip()

    except requests.exceptions.ConnectionError:
        return "Error: Could not connect to Ollama. Is it running? (ollama serve)"
    except requests.exceptions.Timeout:
        return "Error: Request timed out. The model might be slow or overloaded."
    except requests.exceptions.HTTPError as e:
        return f"Error: HTTP {e.response.status_code} - {e.response.text}"
    except KeyError:
        return "Error: Unexpected response format from Ollama. Check model and API."
    except Exception as e:
        return f"Unexpected error: {e}"


def main():
    print("Generating meeting summary...\n")
    summary = generate_summary(MEETING_NOTES)
    print(summary)


if __name__ == "__main__":
    main()

Generating meeting summary...

<think>
Okay, so I need to summarize the meeting notes into three sections: Business Need, Proposed Approach, and Next Step. The client mentioned they want faster onboarding because it currently takes three days. That's their main issue—time inefficiency in the process.

The team suggested using API automation for document verification. That makes sense because automating repetitive tasks can save time. But I should note that compliance approval is still needed, so there might be a hold-up until that part is addressed.

For the next step, they plan to start a pilot program once compliance is handled. It's important to mention that as tentative and subject to change based on how the pilot goes.

I need to keep each section concise and in simple business language. Let me structure it accordingly.
</think>

**Meeting Notes Summary**

1. **Business Need**
   - Client requires faster onboarding process (currently takes 3 days).

2. **Proposed Approach**
   - T

# Chapter 1 Embeddings: Turning Meaning into Numbers

In [2]:
"""
Ollama Embedding-based Retrieval Demo
- Embeds policy document chunks using a local embedding model (nomic-embed-text)
- Computes cosine similarity between a user query and each chunk
- Ranks and displays the most relevant chunks
"""

import requests
import numpy as np
from typing import List, Dict, Tuple

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
OLLAMA_EMBED_URL = "http://localhost:11434/api/embed"
EMBEDDING_MODEL = "nomic-embed-text"   # Must be pulled: ollama pull nomic-embed-text

# Sample document chunks (id, text, source)
DOCUMENT_CHUNKS = [
    {"id": 1, "text": "Travel claims must be submitted within 15 days.", "source": "Travel Policy"},
    {"id": 2, "text": "Late claims require manager approval.", "source": "Travel Policy"},
    {"id": 3, "text": "Employees receive 12 casual leaves per year.", "source": "HR Policy"},
]

USER_QUERY = "How many days do I have to submit travel reimbursement?"


# ----------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> List[float]:
    """
    Fetch embedding vector for a given text from Ollama's embedding API.

    Args:
        text: Input string to embed
        model: Name of the embedding model (must be pulled locally)

    Returns:
        List of floats representing the embedding vector

    Raises:
        requests.exceptions.RequestException: if API call fails
        KeyError: if response JSON structure is unexpected
    """
    payload = {
        "model": model,
        "input": text
    }

    response = requests.post(OLLAMA_EMBED_URL, json=payload, timeout=30)
    response.raise_for_status()          # Raises HTTPError for bad status (4xx, 5xx)

    data = response.json()
    # Expected structure: {"embeddings": [[...]]}
    embedding = data["embeddings"][0]
    return embedding


def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
    """
    Compute cosine similarity between two vectors.

    Args:
        vec_a, vec_b: Lists of equal length

    Returns:
        Cosine similarity in range [-1, 1]
    """
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    # Avoid division by zero (if either vector is all zeros)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot / (norm_a * norm_b))


def rank_chunks(query: str, chunks: List[Dict]) -> List[Tuple[Dict, float]]:
    """
    Embed the query and all chunks, then rank chunks by cosine similarity.

    Args:
        query: User query string
        chunks: List of dicts, each with at least a "text" key

    Returns:
        List of (chunk_dict, similarity_score) sorted descending by score
    """
    # 1. Get query embedding
    query_emb = get_embedding(query)

    # 2. Get embeddings for all chunks (consider adding progress indicator for many chunks)
    chunk_embeddings = []
    for chunk in chunks:
        emb = get_embedding(chunk["text"])
        chunk_embeddings.append(emb)

    # 3. Compute similarities
    scores = [cosine_similarity(query_emb, emb) for emb in chunk_embeddings]

    # 4. Pair chunks with scores and sort
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    return ranked


def display_results(ranked: List[Tuple[Dict, float]], top_k: int = None):
    """
    Pretty-print the top matching chunks.

    Args:
        ranked: Output from rank_chunks()
        top_k: Number of results to show (None = show all)
    """
    if top_k is not None:
        ranked = ranked[:top_k]

    print("\n🔍 Top Matching Chunks:\n")
    for chunk, score in ranked:
        print(f"ID: {chunk['id']}")
        print(f"Source: {chunk['source']}")
        print(f"Text: {chunk['text']}")
        print(f"Similarity: {score:.4f}")
        print("-" * 50)


# ----------------------------------------------------------------------
# Main execution
# ----------------------------------------------------------------------
def main():
    try:
        # Perform retrieval
        ranked_results = rank_chunks(USER_QUERY, DOCUMENT_CHUNKS)

        # Display results (show all in this example)
        display_results(ranked_results)

    except requests.exceptions.ConnectionError:
        print("❌ Error: Could not connect to Ollama. Is it running? (ollama serve)")
    except requests.exceptions.Timeout:
        print("❌ Error: Request to Ollama timed out.")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP error: {e.response.status_code} - {e.response.text}")
    except KeyError as e:
        print(f"❌ Unexpected response format from Ollama. Missing key: {e}")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")


if __name__ == "__main__":
    main()


🔍 Top Matching Chunks:

ID: 1
Source: Travel Policy
Text: Travel claims must be submitted within 15 days.
Similarity: 0.6747
--------------------------------------------------
ID: 2
Source: Travel Policy
Text: Late claims require manager approval.
Similarity: 0.5070
--------------------------------------------------
ID: 3
Source: HR Policy
Text: Employees receive 12 casual leaves per year.
Similarity: 0.5021
--------------------------------------------------


# Chapter 2 Vector Databases: Searching by Meaning

In [3]:
"""
RAG Pipeline with Ollama
- Step 1: Embed documents and user query using a dedicated embedding model (nomic-embed-text)
- Step 2: Retrieve the most relevant chunk via cosine similarity
- Step 3: Generate a grounded answer using a chat model (deepseek-r1:latest)
"""

import requests
import numpy as np
from typing import List, Dict, Tuple

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"      # must be pulled: ollama pull nomic-embed-text
CHAT_MODEL = "deepseek-r1:latest"     # must be pulled: ollama pull deepseek-r1:latest

# Sample document chunks
DOCUMENT_CHUNKS = [
    {"id": 1, "text": "Travel claims must be submitted within 15 days.", "source": "Travel Policy"},
    {"id": 2, "text": "Late claims require manager approval.", "source": "Travel Policy"},
    {"id": 3, "text": "Employees receive 12 casual leaves per year.", "source": "HR Policy"},
]

USER_QUERY = "How many days do I have to submit travel reimbursement?"


# ----------------------------------------------------------------------
# Step 1: Embedding utilities
# ----------------------------------------------------------------------
def get_embedding(text: str, model: str = EMBED_MODEL) -> List[float]:
    """
    Fetch embedding vector from Ollama's /api/embed endpoint.

    Args:
        text: Input string to embed
        model: Name of the embedding model

    Returns:
        List of floats (embedding vector)

    Raises:
        requests.exceptions.RequestException: on network/HTTP errors
        KeyError: if response JSON does not contain 'embeddings'
    """
    url = f"{OLLAMA_BASE_URL}/api/embed"
    payload = {"model": model, "input": text}

    response = requests.post(url, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data["embeddings"][0]


def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
    """
    Compute cosine similarity between two vectors.

    Args:
        vec_a, vec_b: Two lists of equal length

    Returns:
        Cosine similarity (range -1 to 1)
    """
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot / (norm_a * norm_b))


def retrieve_top_chunk(query: str, chunks: List[Dict]) -> Tuple[Dict, float]:
    """
    Find the chunk most similar to the query using embeddings.

    Args:
        query: User question
        chunks: List of dicts with at least a "text" key

    Returns:
        (best_chunk, similarity_score)
    """
    # Embed query and all chunks
    query_emb = get_embedding(query)
    chunk_embs = [get_embedding(chunk["text"]) for chunk in chunks]

    # Compute similarities
    scores = [cosine_similarity(query_emb, emb) for emb in chunk_embs]

    # Find best match
    best_idx = int(np.argmax(scores))
    return chunks[best_idx], scores[best_idx]


# ----------------------------------------------------------------------
# Step 2: Generate answer with chat model
# ----------------------------------------------------------------------
def generate_answer(query: str, context: str, model: str = CHAT_MODEL) -> str:
    """
    Send query + context to an LLM and return the generated answer.

    Args:
        query: User question
        context: Retrieved text chunk to ground the answer
        model: Chat model name

    Returns:
        Generated answer string
    """
    prompt = f"""
Answer the question using only the provided context. If the answer is not found in the context, say: "I do not know from the provided context."

Context: {context}

Question: {query}
"""
    url = f"{OLLAMA_BASE_URL}/api/chat"
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0.0}   # deterministic
    }

    response = requests.post(url, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    return data["message"]["content"].strip()


# ----------------------------------------------------------------------
# Main pipeline
# ----------------------------------------------------------------------
def main():
    try:
        print("🔍 Retrieving most relevant document chunk...")
        best_chunk, score = retrieve_top_chunk(USER_QUERY, DOCUMENT_CHUNKS)

        print("\n📄 Top matching chunk:")
        print(f"ID: {best_chunk['id']} | Source: {best_chunk['source']} | Similarity: {score:.4f}")
        print(f"Text: {best_chunk['text']}\n")

        print("🤖 Generating answer from DeepSeek...")
        answer = generate_answer(USER_QUERY, best_chunk["text"])

        print("\n--- Answer ---")
        print(answer)

    except requests.exceptions.ConnectionError:
        print("❌ Error: Cannot connect to Ollama. Is it running? (ollama serve)")
    except requests.exceptions.Timeout:
        print("❌ Error: Request to Ollama timed out.")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP error: {e.response.status_code} - {e.response.text}")
    except KeyError as e:
        print(f"❌ Unexpected response format from Ollama. Missing key: {e}")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")


if __name__ == "__main__":
    main()

🔍 Retrieving most relevant document chunk...

📄 Top matching chunk:
ID: 1 | Source: Travel Policy | Similarity: 0.6747
Text: Travel claims must be submitted within 15 days.

🤖 Generating answer from DeepSeek...

--- Answer ---
<think>
Okay, so I need to figure out how many days I have to submit my travel reimbursement based on the given context. The context says, "Travel claims must be submitted within 15 days." 

Hmm, let me break this down. The key part here is that it's a requirement to submit the claim within a specific timeframe. So if someone wants to get reimbursed for their travel expenses, they need to do so no later than 15 days from when? Well, usually, such deadlines are counted from the time of the event or the date something happened. But in this case, it's just stated as "within 15 days," without specifying a starting point.

Wait, but regardless of when that period starts, the duration is fixed at 15 days. So if today is day zero, then they have until tomorrow plus 14 m

# Chapter 3 RAG: Retrieve First, Answer Second

In [4]:
"""
Ollama RAG: Retrieve relevant document and answer with DeepSeek
- Embed documents and query using nomic-embed-text
- Retrieve most similar document via cosine similarity
- Generate answer using deepseek-r1:latest
"""

import requests
import numpy as np
from typing import List, Tuple

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"    # must be pulled: ollama pull nomic-embed-text
CHAT_MODEL = "deepseek-r1:latest"   # must be pulled: ollama pull deepseek-r1:latest

# Documents (knowledge base)
DOCUMENTS = [
    "Leave policy: Employees receive 12 casual leaves per year.",
    "Travel policy: Claims must be submitted within 15 days of travel completion.",
    "Insurance policy: Spouse and children are covered under the company health plan.",
]

QUERY = "How many days do I have to submit my travel claim?"


# ----------------------------------------------------------------------
# Embedding utilities
# ----------------------------------------------------------------------
def get_embedding(text: str, model: str = EMBED_MODEL) -> List[float]:
    """
    Get embedding vector from Ollama's /api/embed endpoint.

    Args:
        text: Input string
        model: Embedding model name

    Returns:
        List of floats (embedding vector)

    Raises:
        requests.RequestException: on connection or HTTP error
        KeyError: if response JSON structure is unexpected
    """
    url = f"{OLLAMA_BASE_URL}/api/embed"
    payload = {"model": model, "input": text}

    response = requests.post(url, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data["embeddings"][0]


def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
    """
    Compute cosine similarity between two vectors.

    Args:
        vec_a, vec_b: Two vectors of equal length

    Returns:
        Cosine similarity in range [-1, 1]
    """
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot / (norm_a * norm_b))


# ----------------------------------------------------------------------
# Retrieval
# ----------------------------------------------------------------------
def retrieve_top_document(query: str, documents: List[str]) -> Tuple[str, float]:
    """
    Retrieve the document most similar to the query using embeddings.

    Args:
        query: User question
        documents: List of document strings

    Returns:
        (best_document, similarity_score)
    """
    # Embed query
    query_emb = get_embedding(query)

    # Embed all documents
    doc_embeddings = [get_embedding(doc) for doc in documents]

    # Compute similarities
    scores = [cosine_similarity(query_emb, emb) for emb in doc_embeddings]

    # Find best match
    best_idx = int(np.argmax(scores))
    return documents[best_idx], scores[best_idx]


# ----------------------------------------------------------------------
# Answer generation
# ----------------------------------------------------------------------
def generate_answer(query: str, context: str, model: str = CHAT_MODEL) -> str:
    """
    Ask DeepSeek to answer the query based *only* on the given context.

    Args:
        query: User question
        context: Retrieved document text
        model: Chat model name

    Returns:
        Generated answer string
    """
    prompt = f"""
Answer the question using only the provided context. If the answer is not found, say: "I do not know from the provided context."

Context: {context}

Question: {query}
"""
    url = f"{OLLAMA_BASE_URL}/api/chat"
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0.0}   # deterministic output
    }

    response = requests.post(url, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    return data["message"]["content"].strip()


# ----------------------------------------------------------------------
# Main pipeline
# ----------------------------------------------------------------------
def main():
    try:
        print("🔍 Retrieving most relevant document...")
        top_context, score = retrieve_top_document(QUERY, DOCUMENTS)

        print(f"\n📄 Retrieved Context (similarity: {score:.4f}):")
        print(top_context)

        print("\n🤖 Generating answer from DeepSeek...")
        answer = generate_answer(QUERY, top_context)

        print("\n📝 Answer:")
        print(answer)

    except requests.exceptions.ConnectionError:
        print("❌ Error: Could not connect to Ollama. Is it running? (ollama serve)")
    except requests.exceptions.Timeout:
        print("❌ Error: Request to Ollama timed out.")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP error: {e.response.status_code} - {e.response.text}")
    except KeyError as e:
        print(f"❌ Unexpected response format from Ollama. Missing key: {e}")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")


if __name__ == "__main__":
    main()

🔍 Retrieving most relevant document...

📄 Retrieved Context (similarity: 0.8131):
Travel policy: Claims must be submitted within 15 days of travel completion.

🤖 Generating answer from DeepSeek...

📝 Answer:
<think>
Okay, so I need to figure out how many days I have to submit my travel claim based on the given context. The context says that claims must be submitted within 15 days of travel completion. So, if someone travels and then wants to make a claim for that trip, they have up to 15 days after arriving back or completing their travel to submit it.

I should make sure I'm interpreting this correctly. It's not saying anything about the duration of the trip itself but rather the time frame after the fact when the claim needs to be made. So regardless of how long the trip was, as soon as you finish traveling, 15 days later is the latest you can submit your claim.

I don't see any other information in the context that would change this number, like if there are different deadlines for 

# Chapter 4 Prompt Design: Better Instructions, Better Output

In [7]:
import requests

notes = """
Customer wants faster loan approval. Current process takes 48 hours.
Team suggested document verification API. Compliance approval is mandatory.
"""

good_prompt = f"""
You are a business analyst. Summarize the following meeting notes in exactly 3 sections:
1. Problem
2. Proposed Solution
3. Next Step

Use simple business language.

Notes:
{notes}
"""

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "deepseek-r1:latest",
        "messages": [{"role": "user", "content": good_prompt}],
        "stream": False
    }
)

response.raise_for_status()
print(response.json()["message"]["content"])

<think>
Okay, so I need to summarize the meeting notes into exactly three sections: Problem, Proposed Solution, and Next Step. The problem is that customers are asking for faster loan approvals because the current process takes 48 hours. The team came up with a solution involving a document verification API and compliance approval as a mandatory step.

First, in the Problem section, I should clearly state that customer requests require quicker loan approvals compared to the existing 48-hour timeframe. 

Next, for the Proposed Solution, the team suggested implementing a document verification API which would streamline the process by automating some of the checks. However, it's important to mention that compliance approval is still a mandatory step because it ensures the necessary standards are met.

Finally, the Next Step should outline what needs to happen next, which seems to be testing this new approach with some pilot customers to gather feedback and assess its effectiveness in redu

# Chapter 5 Hallucination Control: Reducing Confident Mistakes

In [8]:
import requests

context = "Travel policy: Claims must be submitted within 15 days of travel completion."

question_1 = "How many days do I have to submit my travel claim?"
question_2 = "What is the yearly insurance premium?"

def ask_safe(question: str):
    prompt = f"""
Answer the question using only the provided context.
If the answer is not found in the context, say exactly: I do not know from the provided context.

Context: {context}

Question: {question}
"""

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "deepseek-r1:latest",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        }
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

print("Q1:", ask_safe(question_1))
print("\nQ2:", ask_safe(question_2))

Q1: <think>
Okay, so the user is asking about how many days they have to submit their travel claim. Let me look at the context provided. The context says that claims must be submitted within 15 days of travel completion. 

Hmm, so it's pretty straightforward here. They just need to make sure they don't wait longer than 15 days after they finish traveling before submitting the claim. I should confirm if there are any other details or exceptions mentioned in the context. The context only provides this one rule, so I can base my answer on that.

I think it's clear enough. There's no mention of varying submission times based on the type of travel or any other factors, so 15 days is the standard timeframe. 

Wait, could there be a situation where they might need more time? The context doesn't say anything about that, so I shouldn't assume extra days unless stated. So my answer should stick to the 15-day window provided.

Okay, confident with that information. Time to put it all together in 

# Chapter 6 Evaluation of LLM Answers: Check Before You Trust

In [9]:
import requests

question = "How many days do I have to submit my travel claim?"
reference = "Travel claims must be submitted within 15 days of travel completion."
answer = "Travel claims must be submitted within 15 days of travel completion."

eval_prompt = f"""
You are evaluating an LLM answer.

Question: {question}
Reference Answer: {reference}
Model Answer: {answer}

Score the Model Answer from 1 to 5 on:
1. Correctness
2. Relevance
3. Completeness
4. Clarity
5. Safety

Then provide one short overall comment.
"""

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "deepseek-r1:latest",
        "messages": [{"role": "user", "content": eval_prompt}],
        "stream": False
    }
)

response.raise_for_status()
print(response.json()["message"]["content"])

<think>
Okay, so I need to evaluate an LLM's answer about how many days you have to submit a travel claim. The question is "How many days do I have to submit my travel claim?" and both the reference answer and the model answer say it must be submitted within 15 days of travel completion.

First, correctness: I guess this means whether the information given is accurate. Both answers are the same, so if that's correct according to the source, then they're both good here.

Relevance: This refers to whether the answer directly addresses the question asked. The model answer clearly states how many days are needed, which matches exactly what was asked. So it should be very relevant.

Completeness: I'm not sure about this one because the response is just a number without any additional details. Maybe if there were more info like conditions or exceptions, it would be complete. But since it's only 15 days, and that's all said, maybe it's somewhat complete but could be improved by adding a bit m

# Chapter 7 Agents vs Workflows: Choosing the Right Architecture

In [11]:
import requests

goal = "Prepare me for tomorrow's client meeting about loan onboarding modernization."

prompt = f"""
You are an AI planning assistant. Given the user goal below, decide whether this should be handled as:
1. a workflow
2. an agent

Then produce a short step-by-step plan.

User Goal: {goal}
"""

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "deepseek-r1:latest",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    }
)

response.raise_for_status()
print(response.json()["message"]["content"])

<think>
Okay, so I need to figure out how to prepare for tomorrow's client meeting about loan onboarding modernization. The user mentioned that the goal is either a workflow or an agent setup. Let me break this down step by step.

First, what exactly is a workflow versus an agent? From what I understand, a workflow is a sequence of tasks that are executed automatically in order, often used to handle repetitive processes efficiently. An agent, on the other hand, is more like an automated assistant that can adapt to different situations and provide personalized responses based on context.

My goal here is to prepare thoroughly for tomorrow's meeting with my client about loan onboarding modernization. I think this involves organizing information, setting up reminders, perhaps tracking key points or questions to discuss, ensuring all materials are ready, etc. That sounds like a structured process that could be automated step by step.

So, starting with the initial preparation, I should pro

# Chapter 8 When Not to Use an LLM: Use Intelligence Wisely

In [12]:
import requests

task = "Check whether customer age is greater than 18."

prompt = f"""
Decide whether the following task should use:
1. simple code / rules
2. SQL / database logic
3. an LLM

Explain the best choice in simple words.

Task: {task}
"""

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "deepseek-r1:latest",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    }
)

response.raise_for_status()
print(response.json()["message"]["content"])

<think>
Okay, so I need to figure out whether to use simple code/rules, SQL/databases, or an LLM for checking if a customer's age is over 18. Let me break this down step by step.

First, what does the task require? It says to check if the customer's age is greater than 18. That sounds straightforward—just compare the age value against 18 and see if it's higher.

Now, thinking about simple code or rules: I remember that for such a basic condition, you can just use an if statement in programming. Like, "if age > 18, then do something." It's really simple and doesn't need anything more complicated than checking a number against another.

On the other hand, SQL is used for querying databases. If I had to get all customers over 18 from a database, I might use an UPDATE or DELETE statement with a WHERE clause. But in this case, the task isn't about retrieving data; it's just a single check, so SQL seems unnecessary.

LLMs are great for understanding text and answering questions, but checking